In [1]:
from datasets import load_dataset

# 加载 IMDB 数据集
dataset = load_dataset("imdb")
# print(dataset)
# DatasetDict({
#     train: Dataset({features: ['text', 'label'], num_rows: 25000})
#     test: Dataset({features: ['text', 'label'], num_rows: 25000})
# })

# 查看样本
# print(dataset["train"][0]["text"][:200])
# print(dataset["train"][0]["label"])  # 0=负面, 1=正面

# 各取 2000 条用于快速实验
# train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
# test_dataset = dataset["test"].shuffle(seed=42).select(range(500))
# print(f"训练集: {len(train_dataset)}, 测试集: {len(test_dataset)}")
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

In [2]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    """对文本进行 tokenize，截断到 256 tokens"""
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

# 批量 tokenize
train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

# 设置 PyTorch 格式
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [3]:
from transformers import AutoModelForSequenceClassification

model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
from transformers import TrainingArguments

In [5]:
training_args = TrainingArguments(
    output_dir="./sentiment_model",  # 模型输出目录
    num_train_epochs=3,  # 训练轮数
    per_device_train_batch_size=16,  # 训练批大小
    per_device_eval_batch_size=32,  # 评估批大小
    learning_rate=2e-5,  # BERT 微调常用学习率
    weight_decay=0.01,  # L2 正则化
    eval_strategy="epoch",  # 每 epoch 评估一次
    save_strategy="epoch",  # 每 epoch 保存一次
    load_best_model_at_end=True,  # 训练结束加载最优模型
    metric_for_best_model="accuracy",  # 以 accuracy 选最优
    logging_steps=50,  # 每 50 步打印日志
    report_to="none",  # 不上报到 wandb 等平台
)

In [6]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

In [8]:
from transformers import Trainer

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.429000,0.314961,0.872000,0.874510
2,0.189900,0.338441,0.870000,0.876190
3,0.123800,0.359183,0.896000,0.896000


TrainOutput(global_step=375, training_loss=0.26643207867940266, metrics={'train_runtime': 117.406, 'train_samples_per_second': 51.105, 'train_steps_per_second': 3.194, 'total_flos': 789333166080000.0, 'train_loss': 0.26643207867940266, 'epoch': 3.0})

In [10]:
trainer.save_model("./sentiment_model/final")
tokenizer.save_pretrained("./sentiment_model/final")